# NSCLC treatment cohort export

Generate routed NSCLC cases until 300 have dated treatment event rows, export them as real OMOP rows, and persist them to a fresh SQLite CDM. This needs the configured HemOnc source database and OMOP vocabulary database.

The run records graph outcomes and schedule rejections instead of silently dropping them. The output path is created under the system temporary directory.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import tempfile

import sqlalchemy as sa
from sqlalchemy.orm import Session
from omop_alchemy.maintenance.cli_schema_tables import create_missing_tables

from _setup import activate_pathway_context, open_routed_profile, require_vocabulary_database
from scoop.config import create_hemonc_engine, get_hemonc_context
from scoop.export.omop import OmopExportConfig, UnmappableComponentError, export_case
from scoop.patients.build import ScheduleBuildOptions, build_case_bundle
from scoop.protocols.hemonc.gateway import SqlHemOncGateway
from scoop.schedule.rollout import UnresolvedScheduleError

require_vocabulary_database()
runtime = open_routed_profile('nsclc', use_database=True)
_, hemonc_config = get_hemonc_context()
hemonc_engine = create_hemonc_engine(hemonc_config)
with hemonc_engine.connect():
    pass
hemonc_session = Session(hemonc_engine)
hemonc_gateway = SqlHemOncGateway(session=hemonc_session)
print(f'loaded {runtime.profile_id}: entry={runtime.entry_graph_name}, tails={len(runtime.tail_contexts)}')
TARGET_CASES = 300
MAX_SEED = 2_000


In [ ]:
accepted = []
outcomes = Counter()
rejections = Counter()
attempted = 0

for seed in range(MAX_SEED):
    if len(accepted) >= TARGET_CASES:
        break
    attempted += 1
    routed = runtime.sample_routed_patient(seed=seed)
    context = activate_pathway_context(runtime, routed)
    try:
        case = build_case_bundle(
            routed.final_sample,
            schedule_options=ScheduleBuildOptions(
                gateway=hemonc_gateway, seed=seed
            ),
        )
        exported = export_case(
            case,
            context.vocabulary_gateway,
            OmopExportConfig(person_id=case.person.person_id),
        )
    except UnresolvedScheduleError as exc:
        outcomes['rejected'] += 1
        rejections[exc.reason] += 1
        continue
    except UnmappableComponentError as exc:
        outcomes['rejected'] += 1
        rejections[f'unmappable {exc.name} ({exc.hemonc_cui})'] += 1
        continue

    treatment = case.treatment_episode
    status = treatment.schedule_status if treatment else None
    if not exported.drug_exposures and not exported.procedures:
        outcomes[status or 'no_schedule'] += 1
        continue
    outcomes['scheduled'] += 1
    accepted.append(exported)

assert len(accepted) == TARGET_CASES, (
    f'Only {len(accepted)} scheduled cases found in {MAX_SEED} seeds'
)
print({
    'attempted': attempted,
    'exported_cases': len(accepted),
    'outcomes': dict(outcomes),
    'rejections': dict(rejections),
})


In [ ]:
cdm_rows = defaultdict(list)
for bundle in accepted:
    if bundle.person is not None:
        cdm_rows['person'].append(bundle.person)
    cdm_rows['condition_occurrence'].extend(bundle.condition_occurrences)
    cdm_rows['measurement'].extend(bundle.measurements)
    cdm_rows['observation'].extend(bundle.observations)
    cdm_rows['drug_exposure'].extend(bundle.drug_exposures)
    cdm_rows['procedure_occurrence'].extend(bundle.procedures)
    cdm_rows['Episode'].extend(bundle.episodes)
    cdm_rows['episode_event'].extend(bundle.episode_events)

CDM_SQLITE_PATH = Path(tempfile.mkdtemp(prefix='scoop-nsclc-treatment-')) / 'cdm.sqlite'
cdm_engine = sa.create_engine(f'sqlite:///{CDM_SQLITE_PATH}')
create_missing_tables(cdm_engine)
with Session(cdm_engine) as session:
    for table_rows in cdm_rows.values():
        session.add_all(table_rows)
    session.commit()

counts = {}
with cdm_engine.connect() as connection:
    for table in sorted(cdm_rows):
        quoted = cdm_engine.dialect.identifier_preparer.quote(table)
        counts[table] = connection.execute(
            sa.text(f'select count(*) from {quoted}')
        ).scalar_one()
print(f'CDM SQLite: {CDM_SQLITE_PATH}')
print(counts)
assert counts['person'] == TARGET_CASES
assert counts['condition_occurrence'] == TARGET_CASES
assert counts['drug_exposure'] + counts['procedure_occurrence'] > 0
